# Lesson 01: Why Capable Agents Still Fail — The Harness Diagnostic Loop

In this notebook, you will:
1. Configure **Google AI Studio Free Tier** (`GEMINI_API_KEY` via `google-genai`) with a deterministic offline fallback.
2. Compare an underspecified prompt against an explicit **Definition of Done** + `AGENTS.md` harness.
3. Compute the **Verification Gap** ($G_{\text{verify}}$) and attribute failures across the 5 harness defense layers.

In [ ]:
!pip install -q -U google-genai

In [ ]:
import os

GEMINI_API_KEY = None
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if GEMINI_API_KEY:
    from google import genai
    client = genai.Client(api_key=GEMINI_API_KEY)
    print("Google AI Studio client initialized (gemini-2.5-flash).")
else:
    client = None
    print("GEMINI_API_KEY not found — running in deterministic offline simulation mode.")

In [ ]:
from dataclasses import dataclass

@dataclass
class FailureAttribution:
    task_id: str
    claimed_done: bool
    verified_pass: bool
    layer: str
    diagnosis: str

def diagnose_prompt(prompt_text: str) -> str:
    if client:
        resp = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=(
                "Classify which of the 5 harness layers is missing in this task prompt "
                "(Task Specification, Context Provision, Execution Environment, Verification Feedback, State Management):\n"
                + prompt_text
            ),
        )
        return resp.text.strip()
    if "Definition of Done" not in prompt_text:
        return "Missing Layer: Task Specification & Verification Feedback (no executable Definition of Done)."
    return "All 5 harness layers addressed."

vague_prompt = "Add a search feature to the app."
harness_prompt = """AGENTS.md: Python 3.11 + FastAPI + SQLAlchemy 2.0.
Definition of Done:
- GET /api/search?q=xxx with pagination (default=20)
- pytest tests/test_search.py passes
- mypy src/ --strict passes"""

print("Vague Prompt Diagnosis:", diagnose_prompt(vague_prompt))
print("Harness Prompt Diagnosis:", diagnose_prompt(harness_prompt))

runs = [
    FailureAttribution("T1", True, False, "Task Specification", "Ambiguous pagination"),
    FailureAttribution("T2", True, False, "Verification Feedback", "No pytest command"),
    FailureAttribution("T3", True, True, "None", "Clean pass"),
]
gap = sum(r.claimed_done for r in runs) / len(runs) - sum(r.verified_pass for r in runs) / len(runs)
print(f"Measured Verification Gap: {gap:.2f}")
assert abs(gap - 0.3333) < 0.01